# TP 3 — Stabiliser puis transformer chaque token
Objectif : implémenter une LayerNorm, comprendre son axe de calcul, puis construire le MLP appliqué séparément à chaque token.

**Livrable** : une LayerNorm pédagogique validée contre `nn.LayerNorm`, un MLP C → 4C → C, et un test de la connexion résiduelle.

In [19]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 : coder LayerNorm

$\hat{x}_{i,j} = \dfrac{x_{i,j} - \mu_i}{\sqrt{\sigma_i^2 + \varepsilon}}, \quad y_{i,j} = \gamma_j \hat{x}_{i,j} + \beta_j$ — i désigne le token ; j, une composante de son embedding.

In [20]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(dim= -1, keepdim=True)
        xvar = x.var(
            dim= -1, keepdim=True, unbiased=False
        )
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta

> **Questions d'interprétation** — Sur quel axe normaliser ? À quoi servent ε, γ et β ?

**Réponses**
- **Axe** : le **dernier** (`dim=-1`), celui des features. Chaque token est normalisé à partir de ses propres composantes, indépendamment des autres tokens et du batch (≠ BatchNorm).
- **ε** évite la division par zéro quand la variance d'un vecteur est (presque) nulle.
- **γ et β** : échelle et décalage **appris** par feature. Sans eux, chaque token serait forcé à moyenne 0 et écart-type 1. Avec eux, le réseau peut réajuster ou même annuler la normalisation si c'est utile. Ils valent 1 et 0 au départ, ce qui donne une normalisation pure.

## Étape 2 : choisir l'axe de normalisation

In [21]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x = torch.randn(32, 100)
x = module(x)

print(x[:, 0].mean(), x[:, 0].std())
print(x[0, :].mean(), x[0, :].std())

tensor(0.1476) tensor(0.8847)
tensor(-2.3842e-09) tensor(1.0050)


> **Questions d'interprétation** — Prédisez quelle ligne donnera environ (0, 1) avant d'exécuter le code.
>
> Validez enfin le résultat avec `torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6)`.

**Réponses**
- **Prédiction** : la **2ᵉ ligne**, `x[0, :]` (les 100 features d'un même token), puisque LayerNorm normalise chaque token sur ses features. Résultat : moyenne ≈ −2·10⁻⁹ ≈ 0, écart-type ≈ 1,005. (Pas exactement 1 car `.std()` divise par N−1 alors que la normalisation divise par N : √(100/99) ≈ 1,005.)
- `x[:, 0]` (une feature à travers 32 tokens) n'est pas contrainte : 0,15 et 0,88 ici.
- `torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6)` → **`True`** : notre implémentation est équivalente à celle de PyTorch (γ=1, β=0, ε=1e-5 et variance biaisée dans les deux cas).

In [22]:
torch.allclose(module(x), nn.LayerNorm(100)(x), atol=1e-6)

True

## Étape 3 : construire le réseau feed-forward

In [23]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

ffwd = FeedForward(100)
delta = ffwd(torch.randn(32, 100))
print(delta.shape)

torch.Size([32, 100])


> **Questions d'interprétation** — Quelle proportion d'activations est positive ? Pourquoi revenir à C ?

**Réponses**
- **Proportion positive** : environ **50 %**. Avec des entrées et des poids aléatoires centrés, les pré-activations sont symétriques autour de 0, et ReLU en met à peu près la moitié à zéro. Vérification : `h = ffwd.net[1](ffwd.net[0](torch.randn(32, 100))); (h > 0).float().mean()`.
- **Revenir à C** : la sortie `delta` est **additionnée** à l'entrée (connexion résiduelle `x + delta`) et le bloc suivant attend des vecteurs de taille C. On ne peut additionner que des tenseurs de même forme. Le passage par 4C ne sert qu'à donner temporairement plus de « place » au calcul.

## Étapes 4 et 5 : ajouter le résiduel
Organisation pré-norm : LayerNorm avant la sous-couche, puis addition résiduelle.

*Note : `B`, `T` et `n_embd` ne sont pas définis dans l'énoncé — choisissez des valeurs.*

In [24]:

B, T, n_embd = 4, 8, 32

layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)
x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta
print(x.shape)
print(delta.shape)
print(x_out.shape)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


**Variantes à tester**
1. Remplacer ReLU par GELU.
2. Tester C → 2C → C, puis C → 8C → C.
3. Compter les paramètres dans chaque cas.
4. Retirer l'addition résiduelle et comparer les formes.

> **Bilan du TP 3** — Quelle dimension LayerNorm conserve-t-elle ? Le MLP mélange-t-il les tokens ? À quoi sert la connexion résiduelle ?

**Réponses — variantes** (avec `n_embd = C = 32`)
1. **GELU au lieu de ReLU** : remplacer `nn.ReLU()` par `nn.GELU()`. Les formes ne changent pas. GELU est une version « lissée » de ReLU qui laisse passer un peu des valeurs négatives (utilisée dans GPT).
2. et 3. **Nombre de paramètres** pour C → hC → C : `(C·hC + hC) + (hC·C + C) = 2hC² + hC + C`. On peut vérifier avec `sum(p.numel() for p in ffwd.parameters())`.

   | variante | paramètres (C = 32) |
   |---|---|
   | C → 2C → C | 4 192 |
   | C → 4C → C | 8 352 |
   | C → 8C → C | 16 672 |

   Le nombre de paramètres est quasiment proportionnel au facteur d'expansion h (doubler h double à peu près les paramètres).
4. **Sans résiduel** (`x_out = delta`) : les formes sont **identiques** `(4, 8, 32)`, rien ne casse. La différence est ailleurs : la sortie ne contient plus l'entrée d'origine, et dans un réseau profond le gradient n'a plus de « chemin direct », ce qui rend l'entraînement beaucoup plus difficile.

**Bilan du TP 3**
- **LayerNorm conserve** toutes les dimensions `(B, T, C)`. Elle ne fait que recentrer et réduire chaque vecteur de taille C.
- **Le MLP ne mélange pas les tokens** : `nn.Linear` agit sur la dernière dimension, donc la même transformation s'applique à chaque token séparément. Seule l'attention fait communiquer les positions.
- **La connexion résiduelle** fait de chaque sous-couche une *correction* `x + Δx` plutôt qu'un remplacement. L'information d'origine est préservée et le gradient peut traverser directement tous les blocs, ce qui stabilise l'entraînement des réseaux profonds.